In [2]:
import os
import json
import cv2
import random
from glob import glob
from tqdm import tqdm
from multiprocessing import Pool, cpu_count, freeze_support

DIRS_TO_PROCESS = [
    {"img": "dataset/images", "lbl": "dataset/annotations"},
    {"img": "unity/images", "lbl": "unity/labels"}
]

YOLO_DIR = "final_dataset_yoloOnly"

CLASS_MAP = {
    '0': 0, '1': 1, '2': 2, '3': 3, '4': 4,
    '5': 5, '6': 6, '7': 7, '8': 8, '9': 9,
    'minus': 10,
    'point': 11,
    'tag': 12
}

def convert_to_yolo_bbox(box, img_w, img_h):
    dw = 1. / img_w
    dh = 1. / img_h
    x = (box[0] + box[2]) / 2.0
    y = (box[1] + box[3]) / 2.0
    w = box[2] - box[0]
    h = box[3] - box[1]
    x, w, y, h = x * dw, w * dw, y * dh, h * dh
    return x, y, w, h

def process_single_pair(args):
    img_path, json_path, split_name, yolo_root = args
    local_counts = {}

    try:
        img = cv2.imread(img_path)
        if img is None: return {}
        h, w, _ = img.shape

        with open(json_path, 'r') as f:
            data = json.load(f)

        base_name = os.path.splitext(os.path.basename(img_path))[0]

        dst_img = os.path.join(yolo_root, "images", split_name, base_name + ".jpg")
        cv2.imwrite(dst_img, img)

        yolo_txt_path = os.path.join(yolo_root, "labels", split_name, base_name + ".txt")

        with open(yolo_txt_path, 'w') as f_yolo:
            for item in data:
                chars = item.get("chars", [])
                for char_obj in chars:
                    cls_type = char_obj.get("class_name") or char_obj.get("class")
                    char_val = char_obj.get("char_val") or char_obj.get("char")
                    bbox = char_obj.get("bbox")

                    if not bbox: continue

                    class_id = -1
                    if cls_type == "digit" and char_val in CLASS_MAP:
                        class_id = CLASS_MAP[char_val]
                    elif cls_type == "minus":
                        class_id = CLASS_MAP["minus"]
                    elif cls_type == "point":
                        class_id = CLASS_MAP["point"]
                    elif cls_type == "tag":
                        class_id = CLASS_MAP["tag"]

                    if class_id != -1:
                        yx, yy, yw, yh = convert_to_yolo_bbox(bbox, w, h)
                        if yw > 0 and yh > 0:
                            f_yolo.write(f"{class_id} {yx:.6f} {yy:.6f} {yw:.6f} {yh:.6f}\n")
                            k = str(class_id)
                            local_counts[k] = local_counts.get(k, 0) + 1

    except Exception as e:
        return {}

    return local_counts

if __name__ == '__main__':
    freeze_support()

    for split in ["train", "val", "test"]:
        os.makedirs(os.path.join(YOLO_DIR, "images", split), exist_ok=True)
        os.makedirs(os.path.join(YOLO_DIR, "labels", split), exist_ok=True)

    all_pairs = []
    print("Збираємо файли...")
    for d in DIRS_TO_PROCESS:
        images = glob(os.path.join(d["img"], "*.jpg")) + glob(os.path.join(d["img"], "*.png"))
        for img_path in images:
            fname = os.path.basename(img_path)
            json_path = os.path.join(d["lbl"], os.path.splitext(fname)[0] + ".json")
            if os.path.exists(json_path):
                all_pairs.append((img_path, json_path))

    random.seed(42)
    random.shuffle(all_pairs)
    train_end = int(len(all_pairs) * 0.8)
    val_end = int(len(all_pairs) * 0.9)
    splits = {
        "train": all_pairs[:train_end],
        "val": all_pairs[train_end:val_end],
        "test": all_pairs[val_end:]
    }

    tasks = []
    for split_name, pairs in splits.items():
        for img, js in pairs:
            tasks.append((img, js, split_name, YOLO_DIR))

    print(f"Обробка {len(all_pairs)} файлів...")

    final_counts = {}
    with Pool(cpu_count()) as pool:
        results = list(tqdm(pool.imap_unordered(process_single_pair, tasks), total=len(tasks)))

    for res in results:
        for k, v in res.items():
            final_counts[k] = final_counts.get(k, 0) + v

    yaml_content = f"""
path: {os.path.abspath(YOLO_DIR)}
train: images/train
val: images/val
test: images/test

names:
  0: '0'
  1: '1'
  2: '2'
  3: '3'
  4: '4'
  5: '5'
  6: '6'
  7: '7'
  8: '8'
  9: '9'
  10: minus
  11: point
  12: tag
"""
    with open(os.path.join(YOLO_DIR, "dataset_yolo.yaml"), 'w') as f:
        f.write(yaml_content)

    print("\n✅ Готово! Статистика класів:")
    for k in sorted(final_counts.keys(), key=lambda x: int(x)):
        print(f"  Class {k}: {final_counts[k]}")

🔍 Збираємо файли...
🚀 Обробка 6988 файлів...


100%|███████████████████████████████████████████████████████████████████████████████| 6988/6988 [01:32<00:00, 75.56it/s]


✅ Готово! Статистика класів:
  Class 0: 7434
  Class 1: 9728
  Class 2: 9263
  Class 3: 8504
  Class 4: 10526
  Class 5: 9691
  Class 6: 7063
  Class 7: 7839
  Class 8: 7222
  Class 9: 8213
  Class 10: 2264
  Class 11: 8962
  Class 12: 37482
